In [ ]:
from utils.postgres_util import get_engine_from_env, read_sql_dataframe
from utils.timestamp_stage_writer import (
    ensure_simulation_timing_config_table,
    insert_simulation_timing_config,
)


In [ ]:

engine = get_engine_from_env()


In [ ]:

ensure_simulation_timing_config_table(
    engine=engine,
    schema="capstone",
    table_name="simulation_timing_config",
)


In [ ]:

insert_simulation_timing_config(
    engine=engine,
    dataset_id="pump_synthetic_v1",
    run_id="premelt_run_001",
    simulation_start_datetime="2026-03-19 08:00:00+00:00",
    sampling_interval_seconds=1.0,
    schema="capstone",
    table_name="simulation_timing_config",
    set_active=True,
    deactivate_existing_for_run=True,
)


In [ ]:

print("Timing config ready.")

----

In [ ]:

sample_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        observation_timestamp,
        message_sequence_index,
        sensor_name,
        sensor_index,
        sensor_value,
        stream_state,
        phase,
        meta_episode_id
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    ORDER BY observation_index, message_sequence_index, sensor_index
    LIMIT 104
    """
)

display(sample_dataframe)

In [ ]:
timestamp_check_dataframe = read_sql_dataframe(
    engine,
    """
    SELECT
        observation_index,
        COUNT(*) AS message_count,
        COUNT(DISTINCT observation_timestamp) AS distinct_timestamps_in_observation,
        MIN(observation_timestamp) AS observation_timestamp_min,
        MAX(observation_timestamp) AS observation_timestamp_max
    FROM capstone.synthetic_sensor_messages_timestamped_stage
    GROUP BY observation_index
    ORDER BY observation_index
    LIMIT 10
    """
)

display(timestamp_check_dataframe)